In [1]:
from pathlib import Path
p = Path(r"C:\Users\sbhat\Data Mining\ratings.csv")
print("Exists:", p.exists())
print("Is file:", p.is_file())










Exists: True
Is file: True


In [2]:
import pandas as pd
ratings = pd.read_csv(r"C:\Users\sbhat\Data Mining\ratings.csv")
print(ratings.shape)
ratings.head()


(26024289, 4)


,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556


In [3]:
# --- Task 2: Recommender Systems (Collaborative Filtering + PMF) ---

import os
import math
import time
import numpy as np
import pandas as pd

from pathlib import Path
from collections import defaultdict

# Surprise (installed in your `recsys` env)
from surprise import Dataset, Reader
from surprise import KNNBasic, SVD
from surprise.model_selection import cross_validate, KFold, train_test_split
from surprise import accuracy

# ----------------- USER CONFIG -----------------
DATA_PATH = Path(r"C:\Users\sbhat\Data Mining\ratings.csv")  # <- keep this path
N_ROWS = 200_000   # try a small sample first (set to None to use full 26M)
TEST_SIZE = 0.2    # 80/20 train/test split
RANDOM_STATE = 42
N_JOBS = -1        # parallel folds if Surprise supports it on your platform
# ------------------------------------------------

np.random.seed(RANDOM_STATE)

# Output folder
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Using file:", DATA_PATH)
print("Sample rows:", N_ROWS)


Using file: C:\Users\sbhat\Data Mining\ratings.csv
Sample rows: 200000


In [4]:
usecols = ["userId", "movieId", "rating", "timestamp"]  # MovieLens columns
dtypes = {"userId": np.int32, "movieId": np.int32, "rating": np.float32, "timestamp": np.int64}

if N_ROWS is None:
    ratings_df = pd.read_csv(DATA_PATH, usecols=usecols, dtype=dtypes)
else:
    # Efficient first-N load to avoid reading the whole 26M in memory
    ratings_df = pd.read_csv(DATA_PATH, usecols=usecols, nrows=N_ROWS, dtype=dtypes)

print(ratings_df.shape)
ratings_df.head()


(200000, 4)


,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556


In [5]:
# Surprise expects: user, item, rating with a Reader that knows rating scale.
# MovieLens ratings are typically from 0.5 to 5.0 (step 0.5). Adjust if yours differ.
min_r, max_r = float(ratings_df["rating"].min()), float(ratings_df["rating"].max())
reader = Reader(rating_scale=(min_r, max_r))

data = Dataset.load_from_df(
    ratings_df[["userId", "movieId", "rating"]],
    reader
)


In [6]:
def knn_model(sim_name: str, user_based: bool):
    # KNNBasic is a solid baseline CF in Surprise
    sim_options = {"name": sim_name, "user_based": user_based}
    return KNNBasic(sim_options=sim_options, verbose=False)

def svd_model():
    # SVD with biases (the common Surprise MF)
    return SVD(random_state=RANDOM_STATE)

def pmf_model():
    # Probabilistic Matrix Factorization = SVD without biases
    return SVD(biased=False, random_state=RANDOM_STATE)

models = []
# User-CF
for sim in ["cosine", "msd", "pearson"]:
    models.append(("UserCF-"+sim, knn_model(sim, user_based=True)))
# Item-CF
for sim in ["cosine", "msd", "pearson"]:
    models.append(("ItemCF-"+sim, knn_model(sim, user_based=False)))

# MF family
models.append(("SVD", svd_model()))
models.append(("PMF", pmf_model()))

models


[('UserCF-cosine',
  <surprise.prediction_algorithms.knns.KNNBasic at 0x1f18758a980>),
 ('UserCF-msd',
  <surprise.prediction_algorithms.knns.KNNBasic at 0x1f187589b10>),
 ('UserCF-pearson',
  <surprise.prediction_algorithms.knns.KNNBasic at 0x1f18758bb80>),
 ('ItemCF-cosine',
  <surprise.prediction_algorithms.knns.KNNBasic at 0x1f18758ab00>),
 ('ItemCF-msd',
  <surprise.prediction_algorithms.knns.KNNBasic at 0x1f18758bc40>),
 ('ItemCF-pearson',
  <surprise.prediction_algorithms.knns.KNNBasic at 0x1f18758bca0>),
 ('SVD',
  <surprise.prediction_algorithms.matrix_factorization.SVD at 0x1f187589720>),
 ('PMF',
  <surprise.prediction_algorithms.matrix_factorization.SVD at 0x1f18758a440>)]

In [7]:
cv_results = []
kf = KFold(n_splits=5, random_state=RANDOM_STATE, shuffle=True)

for name, algo in models:
    t0 = time.time()
    cv = cross_validate(
        algo,
        data,
        measures=["RMSE", "MAE"],
        cv=kf,
        n_jobs=N_JOBS,
        verbose=False
    )
    elapsed = time.time() - t0
    cv_results.append({
        "model": name,
        "rmse_mean": float(np.mean(cv["test_rmse"])),
        "rmse_std": float(np.std(cv["test_rmse"])),
        "mae_mean": float(np.mean(cv["test_mae"])),
        "mae_std": float(np.std(cv["test_mae"])),
        "fit_time_mean": float(np.mean(cv["fit_time"])),
        "test_time_mean": float(np.mean(cv["test_time"])),
        "elapsed_sec_total": float(elapsed),
        "n_rows": int(len(ratings_df)),
    })
    print(f"{name:12s} | RMSE {np.mean(cv['test_rmse']):.4f} | MAE {np.mean(cv['test_mae']):.4f} | rows={len(ratings_df)}")

val_df = pd.DataFrame(cv_results).sort_values("rmse_mean").reset_index(drop=True)
val_path = OUT_DIR / "validation_report.csv"
val_df.to_csv(val_path, index=False)
val_df


UserCF-cosine | RMSE 0.9566 | MAE 0.7369 | rows=200000
UserCF-msd   | RMSE 0.9193 | MAE 0.7030 | rows=200000
UserCF-pearson | RMSE 0.9562 | MAE 0.7398 | rows=200000
ItemCF-cosine | RMSE 0.9583 | MAE 0.7411 | rows=200000
ItemCF-msd   | RMSE 0.9062 | MAE 0.6957 | rows=200000
ItemCF-pearson | RMSE 0.9533 | MAE 0.7369 | rows=200000
SVD          | RMSE 0.8662 | MAE 0.6634 | rows=200000
PMF          | RMSE 0.9707 | MAE 0.7379 | rows=200000


,model,rmse_mean,rmse_std,mae_mean,mae_std,fit_time_mean,test_time_mean,elapsed_sec_total,n_rows
0,SVD,0.866198,0.004375,0.663418,0.002925,1.776919,0.342752,5.677547,200000
1,ItemCF-msd,0.906246,0.004975,0.695671,0.004088,8.697058,12.734841,25.042720,200000
2,UserCF-msd,0.919325,0.002956,0.703022,0.001937,2.361203,5.920867,11.825869,200000
3,ItemCF-pearson,0.953280,0.004856,0.736881,0.004043,27.788552,12.476777,43.900801,200000
4,UserCF-pearson,0.956199,0.002980,0.739793,0.001961,4.786301,5.534123,13.845733,200000
5,UserCF-cosine,0.956606,0.003086,0.736905,0.001883,4.308597,5.746882,14.634794,200000
6,ItemCF-cosine,0.958325,0.005253,0.741068,0.004463,16.932269,12.013016,32.607637,200000
7,PMF,0.970747,0.003039,0.737889,0.002074,1.801494,0.312155,5.778758,200000


In [8]:
# Build full Surprise dataset again (train/test split works from full dataset object)
trainset, testset = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE)

test_rows = []
test_report = []

for name, algo in models:
    t0 = time.time()
    algo.fit(trainset)
    preds = algo.test(testset)
    rmse = accuracy.rmse(preds, verbose=False)
    mae = accuracy.mae(preds, verbose=False)
    elapsed = time.time() - t0

    # Store per-row predictions
    for p in preds:
        test_rows.append({
            "model": name,
            "userId": p.uid,
            "movieId": p.iid,
            "true_rating": p.r_ui,
            "est_rating": p.est
        })

    test_report.append({
        "model": name,
        "rmse": float(rmse),
        "mae": float(mae),
        "elapsed_sec": float(elapsed),
        "n_rows": int(len(ratings_df)),
        "test_size": float(TEST_SIZE),
    })
    print(f"{name:12s} | TEST RMSE {rmse:.4f} | MAE {mae:.4f}")

# Save predictions
preds_df = pd.DataFrame(test_rows)
preds_path = OUT_DIR / "test_predictions.csv"
preds_df.to_csv(preds_path, index=False)

# Save test report
test_df = pd.DataFrame(test_report).sort_values("rmse").reset_index(drop=True)
test_path = OUT_DIR / "test_report.csv"
test_df.to_csv(test_path, index=False)

test_df


UserCF-cosine | TEST RMSE 0.9577 | MAE 0.7364
UserCF-msd   | TEST RMSE 0.9191 | MAE 0.7013
UserCF-pearson | TEST RMSE 0.9560 | MAE 0.7377
ItemCF-cosine | TEST RMSE 0.9502 | MAE 0.7335
ItemCF-msd   | TEST RMSE 0.8992 | MAE 0.6890
ItemCF-pearson | TEST RMSE 0.9456 | MAE 0.7302
SVD          | TEST RMSE 0.8596 | MAE 0.6585
PMF          | TEST RMSE 0.9665 | MAE 0.7345


,model,rmse,mae,elapsed_sec,n_rows,test_size
0,SVD,0.859571,0.658481,3.187339,200000,0.2
1,ItemCF-msd,0.899202,0.689027,28.113625,200000,0.2
2,UserCF-msd,0.919108,0.701343,9.939742,200000,0.2
3,ItemCF-pearson,0.945563,0.730238,37.157229,200000,0.2
4,ItemCF-cosine,0.950150,0.733480,35.150150,200000,0.2
5,UserCF-pearson,0.955987,0.737746,13.067906,200000,0.2
6,UserCF-cosine,0.957730,0.736381,12.573345,200000,0.2
7,PMF,0.966518,0.734466,3.099703,200000,0.2


In [9]:
# Build a short summary of the best models on CV and Test
best_cv = val_df.sort_values("rmse_mean").iloc[0]
best_test = test_df.sort_values("rmse").iloc[0]

lines = []
lines.append("TASK 2 – Recommender System: Summary\n")
lines.append(f"Dataset rows used: {len(ratings_df)}")
lines.append(f"Train/Test split: {int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)}")
lines.append("")

lines.append("Cross-Validation (5-fold):")
lines.append(f"  Best model: {best_cv['model']}")
lines.append(f"  RMSE (mean): {best_cv['rmse_mean']:.4f} ± {best_cv['rmse_std']:.4f}")
lines.append(f"  MAE (mean):  {best_cv['mae_mean']:.4f} ± {best_cv['mae_std']:.4f}")
lines.append("")

lines.append("Held-out Test:")
lines.append(f"  Best model: {best_test['model']}")
lines.append(f"  RMSE: {best_test['rmse']:.4f}")
lines.append(f"  MAE:  {best_test['mae']:.4f}")
lines.append("")

lines.append("Notes:")
lines.append("- UserCF vs ItemCF: different similarity measures (cosine, MSD, Pearson) can shift performance.")
lines.append("- SVD is MF with biases; PMF is SVD with biased=False (pure matrix factorization).")
lines.append("- With very large data, try running with a sample (N_ROWS) first, then scale up.")
lines.append("- Lower RMSE/MAE is better.")

summary = "\n".join(lines)
sum_path = OUT_DIR / "summary.txt"
with open(sum_path, "w", encoding="utf-8") as f:
    f.write(summary)

print(summary)



TASK 2 – Recommender System: Summary

Dataset rows used: 200000
Train/Test split: 80/20

Cross-Validation (5-fold):
  Best model: SVD
  RMSE (mean): 0.8662 ± 0.0044
  MAE (mean):  0.6634 ± 0.0029

Held-out Test:
  Best model: SVD
  RMSE: 0.8596
  MAE:  0.6585

Notes:
- UserCF vs ItemCF: different similarity measures (cosine, MSD, Pearson) can shift performance.
- SVD is MF with biases; PMF is SVD with biased=False (pure matrix factorization).
- With very large data, try running with a sample (N_ROWS) first, then scale up.
- Lower RMSE/MAE is better.
